# PK/PD Antibiotic Model: Example Workflow

This notebook demonstrates how to use the PK/PD modeling framework to:

- run a baseline simulation

- visualize antibiotic concentration and bacterial burden over time

- perform a parameter sweep

- compare treatment outcomes across conditions

The core model code is imported from `src/pkpd_model.py`.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Add the parent directory to the Python path so we can import from src/

# This is necessary because the notebook is inside the "notebooks" folder

sys.path.append(os.path.abspath(".."))

# Import core model functions

from src.pkpd_model import *

In [ ]:
## INITIALIZE MODEL PARAMETERS ##

# Create default model parameters (PK, PD, and biological parameters)
params = create_params()

# Create default simulation settings (dose, interval, simulation time, etc.)
sim_settings = create_sim_settings()

# Display the parameter dictionaries
params, sim_settings

In [ ]:
## Run a single PK/PD simulation using default parameters ##

# Outputs:
# t = time points
# C = drug concentration over time
# B = bacterial burden over time
t, C, B = run_simulation(params = params, sim_settings = sim_settings)

# Print final bacterial burden for quick reference
print(f"Final bacterial burden: {B[-1]:.2e}")

In [ ]:
## PLOT PK/PD PROFILES ##

# Create a figure with two subplots: PK (left) and PD (right)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot PK: drug concentration over time
axes[0].plot(t, C)
axes[0].set_title("PK Profile")
axes[0].set_xlabel("Time (hours)")
axes[0].set_ylabel("Concentration (mg/L)")

# Plot PD: bacterial burden over time (log scale for clarity)
axes[1].plot(t, B)
axes[1].set_yscale("log")
axes[1].set_title("PD Profile")
axes[1].set_xlabel("Time (hours)")
axes[1].set_ylabel("Bacterial Burden (CFU/mL)")

# Improve layout
plt.tight_layout()
plt.show()

In [ ]:
## COMPUTE AND PLOT LOG REDUCTION ##

# Set a detection limit to avoid log(0) issues
DETECTION_LIMIT = 1.0

# Ensure bacterial burden never goes below detection limit
B_safe = np.maximum(B, DETECTION_LIMIT)

# Initial bacterial burden
B0 = B_safe[0]

# Compute log10 reduction over time

# This represents how many orders of magnitude the bacteria have been reduced
log_reduction = np.log10(B0 / B_safe)

# Plot antibacterial effect
plt.plot(t, log_reduction)

# Add reference line for 3-log reduction (bactericidal threshold)
plt.axhline(3, linestyle="--", color="gray")
plt.xlabel("Time (hours)")
plt.ylabel("Log10 Reduction")
plt.title("Antibacterial Effect")
plt.show()